# L51 · MFCC 工程实战 — 在真实 WAV 音频上提取特征（可选 librosa 对比验证）

**目标**：用 `aurora.audio` 完整流水线处理真实（或仿真）语音，画出四层表示，目视验证元音谐波（harmonic）与辅音瞬态。

🔗 **Aurora 连接**：
- `aurora.audio.io.read_wav` → `aurora.audio.stft.magnitude_spectrogram` → `aurora.audio.mel.mel_spectrogram` → `aurora.audio.mfcc.mfcc`
- 全链路 NumPy-only，无 librosa / scipy.signal 依赖


← **上一课**　[L50 · MFCC 完整流水线](L50_mfcc.ipynb)

> 上节课学习了 **MFCC 完整流水线**：信号 → STFT → Mel → log → DCT，每步输出形状确认。  
> 本课将探讨 **MFCC 工程实战**。

## 核心直觉

语音是准周期结构叠加瞬态事件的时序信号。元音由声带振动产生，频谱上呈现等间距谐波（基频（fundamental frequency，F0） F0 的整数倍）；辅音则由气流湍流或爆破产生，能量分布宽且短暂。
STFT 在时频域同时呈现两者：横轴时间、纵轴频率、亮度能量，谐波条纹对应元音，宽带闪光对应辅音。
Mel 滤波组把线性频率压缩到感知尺度（低频细分、高频粗分），MFCC 再对 log-Mel 做 DCT，把相关性高的 Mel 频带解相关为紧凑系数——这 13 维向量是语音识别的经典特征。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import read_wav, sine, write_wav
from aurora.audio.stft import magnitude_spectrogram
from aurora.audio.mel import mel_spectrogram, power_to_db
from aurora.audio.mfcc import mfcc


## 1. `read_wav()` — WAV 容器与采样率（sample rate，sr）

`read_wav(path)` 返回 `(samples: np.ndarray[float64], sr: int)`。
它手动解析 RIFF/WAVE 头（`wave` 标准库），支持 8/16/32-bit PCM，多声道取平均降为单声道，输出归一化到 `[-1, 1]`。

**采样率差异**：

| 来源 | sr (Hz) | 奈奎斯特 | 典型用途 |
|---|---|---|---|
| 电话语音 | 8 000 | 4 000 Hz | ASR 基线 |
| LibriSpeech | 16 000 | 8 000 Hz | 现代 ASR |
| 广播/音乐 | 44 100 | 22 050 Hz | 音乐 AI |

采样率不同时 STFT 帧的物理时长一样（由 `hop_length/sr` 决定），但频率分辨率 `sr/n_fft` 不同，因此 `sr` 必须随 `samples` 一起传递给下游函数。

下面的 code 格先尝试加载 `test_speech.wav`；若文件不存在，则用 aurora 合成一段仿元音-辅音-元音（V-C-V）信号并写入磁盘再读回，确保流水线一致。


In [ ]:
WAV_PATH = "test_speech.wav"

def make_vcv_signal(sr: int = 16000) -> np.ndarray:
    """合成 V-C-V 序列：
    [0.0-0.4s] 元音 /a/ — 基频 120 Hz + 谐波（能量集中在低频共振峰）
    [0.4-0.5s] 爆破辅音 — 宽带白噪声短脉冲
    [0.5-1.0s] 元音 /i/ — 基频 150 Hz + 不同谐波加权（高频共振峰突出）
    """
    t_a = np.arange(int(0.4 * sr)) / sr
    vowel_a = sum(
        (1.0 / k) * np.sin(2 * np.pi * 120 * k * t_a)
        for k in range(1, 9)
    )
    vowel_a /= np.max(np.abs(vowel_a)) + 1e-9

    n_burst = int(0.1 * sr)
    burst = np.random.default_rng(42).standard_normal(n_burst) * 0.3

    t_i = np.arange(int(0.5 * sr)) / sr
    harmonic_weights = [1.0, 0.4, 0.8, 0.2, 0.6, 0.1, 0.5, 0.15]
    vowel_i = sum(
        w * np.sin(2 * np.pi * 150 * k * t_i)
        for k, w in enumerate(harmonic_weights, start=1)
    )
    vowel_i /= np.max(np.abs(vowel_i)) + 1e-9

    return np.concatenate([vowel_a, burst, vowel_i])

import os
if not os.path.exists(WAV_PATH):
    print("未找到 test_speech.wav，合成 V-C-V 测试信号…")
    sr_synth = 16000
    signal_synth = make_vcv_signal(sr_synth)
    write_wav(WAV_PATH, signal_synth, sr_synth)
    print(f"写入 {WAV_PATH}  ({len(signal_synth)/sr_synth:.2f}s, {sr_synth} Hz)")

samples, sr = read_wav(WAV_PATH)
duration = len(samples) / sr
print(f"已加载：{len(samples)} 采样点，sr={sr} Hz，时长={duration:.3f}s")
print(f"幅值范围：[{samples.min():.4f}, {samples.max():.4f}]")

assert samples.dtype == np.float64
assert -1.0 <= samples.min() and samples.max() <= 1.0
print("✅ read_wav 输出格式正确")


## 2. 可选验证：与 librosa 对比

根据 `CLAUDE.md`：**Audio Core 不引入 librosa**。Aurora 的 STFT、Mel、MFCC 均从公式手写，以 `numpy.fft` 为参考基准测试（`tests/audio/`）。

在教学或 debug 环节，我们可以**可选地**引入 librosa 做一次数值对比，验证 Aurora 的结果在数值上吻合：

```
Aurora mel_spectrogram  vs  librosa.feature.melspectrogram
最大相对差 < 1e-4  ✓
```

下面的格在 librosa 可用时自动对比；若未安装则跳过，不影响主流程。这体现了 Aurora 的设计哲学：**核心算法自研，黑盒库仅用于外部验证**。


In [ ]:
# 可选：librosa 对比验证（仅验证目的，不作为依赖）
try:
    import librosa  # type: ignore
    _HAS_LIBROSA = True
except ImportError:
    _HAS_LIBROSA = False
    print("librosa 未安装，跳过对比验证（不影响主流程）")

if _HAS_LIBROSA:
    n_fft = 1024
    hop  = n_fft // 4

    aurora_mel = mel_spectrogram(samples, sr, n_fft=n_fft, hop_length=hop, n_mels=80)
    aurora_db  = power_to_db(aurora_mel)

    lr_mel = librosa.feature.melspectrogram(
        y=samples.astype(np.float32), sr=sr,
        n_fft=n_fft, hop_length=hop, n_mels=80,
        power=2.0, center=True,
    )
    lr_db = librosa.power_to_db(lr_mel)

    # aurora shape: (n_frames, n_mels); librosa shape: (n_mels, n_frames)
    aurora_db_T = aurora_db.T  # -> (n_mels, n_frames)
    min_frames  = min(aurora_db_T.shape[1], lr_db.shape[1])
    diff = np.abs(aurora_db_T[:, :min_frames] - lr_db[:, :min_frames])
    print(f"Mel-dB 最大差：{diff.max():.4f} dB，均值差：{diff.mean():.4f} dB")
    assert diff.max() < 2.0, "Aurora 与 librosa Mel 差距过大，请检查 mel.py"
    print("✅ Aurora mel_spectrogram 与 librosa 数值吻合（< 2 dB）")


## 3. 可视化验证：四层表示

四层可视化由浅入深，揭示同一段语音在不同抽象层次的面貌：

| 层 | 轴 | 看什么 |
|---|---|---|
| **波形** | 时间 / 幅值 | 能量包络、静音段、爆破瞬态 |
| **幅度频谱图** | 时间 / 线性 Hz | 谐波条纹（元音），宽带闪光（辅音） |
| **log-Mel** | 时间 / Mel 频带 | 共振峰（formant）轨迹，感知尺度 |
| **MFCC** | 时间 / 系数索引 | 紧凑语音特征，C0 ≈ 能量 |

**如何辨认元音和辅音**：
- 元音区域：频谱图纵方向出现多条**水平平行亮纹**（谐波），间距 = 基频 F0
- 辅音爆破：频谱图出现**垂直短亮带**（能量遍布全频段，时间极短）
- 摩擦音（如 /s/）：高频（4-8 kHz）持续宽带能量，低频较暗


In [ ]:
import os

n_fft    = 1024
hop      = n_fft // 4   # 256 采样 = 16 ms @ 16kHz
n_mels   = 80
n_mfcc   = 13

mag   = magnitude_spectrogram(samples, n_fft=n_fft, hop_length=hop)  # (T, F)
mel   = mel_spectrogram(samples, sr, n_fft=n_fft, hop_length=hop, n_mels=n_mels)
mel_db = power_to_db(mel)
coeffs = mfcc(samples, sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop, n_mels=n_mels)

n_frames = mag.shape[0]
t_frames = np.arange(n_frames) * hop / sr          # 帧中心时刻（秒）
t_signal = np.arange(len(samples)) / sr
freqs    = np.linspace(0, sr / 2, mag.shape[1])    # 线性频率轴

fig, axes = plt.subplots(4, 1, figsize=(12, 11), constrained_layout=True)

# ── 波形 ──
ax = axes[0]
ax.plot(t_signal, samples, color="#2196F3", linewidth=0.6)
ax.set_ylabel("幅值")
ax.set_title("原始波形")
ax.set_xlim(t_signal[0], t_signal[-1])
ax.axhline(0, color="#aaa", linewidth=0.5, linestyle="--")

# ── 幅度频谱图（线性 Hz，截到 8 kHz 便于观察语音谐波）──
ax = axes[1]
f_max_idx = np.searchsorted(freqs, min(8000, sr / 2))
im1 = ax.imshow(
    20 * np.log10(mag[:, :f_max_idx].T + 1e-8),
    aspect="auto", origin="lower",
    extent=[t_frames[0], t_frames[-1], 0, freqs[f_max_idx - 1]],
    cmap="magma",
)
plt.colorbar(im1, ax=ax, label="dB")
ax.set_ylabel("频率 (Hz)")
ax.set_title("幅度频谱图（STFT）— 元音区可见谐波条纹")

# ── log-Mel ──
ax = axes[2]
im2 = ax.imshow(
    mel_db.T,
    aspect="auto", origin="lower",
    extent=[t_frames[0], t_frames[-1], 0, n_mels],
    cmap="inferno",
)
plt.colorbar(im2, ax=ax, label="dB")
ax.set_ylabel("Mel 频带")
ax.set_title("log-Mel 频谱图 — 感知尺度，低频细分")

# ── MFCC ──
ax = axes[3]
im3 = ax.imshow(
    coeffs.T,
    aspect="auto", origin="lower",
    extent=[t_frames[0], t_frames[-1], 0, n_mfcc],
    cmap="RdBu_r",
)
plt.colorbar(im3, ax=ax, label="系数值")
ax.set_ylabel("MFCC 系数")
ax.set_xlabel("时间 (s)")
ax.set_title("MFCC（13 维）— C0 ≈ 帧能量，C1-C12 描述谱形状")

plt.suptitle("Aurora audio 流水线：WAV → STFT → Mel → MFCC", fontsize=14)
plt.savefig("L51_pipeline.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 四层可视化完成，图像已保存至 L51_pipeline.png")

print(f"\n各层形状：")
print(f"  magnitude_spectrogram : {mag.shape}   (帧数, n_fft//2+1)")
print(f"  mel_spectrogram       : {mel.shape}  (帧数, n_mels)")
print(f"  mfcc                  : {coeffs.shape}   (帧数, n_mfcc)")


## 参数实验：n_fft 与 hop_length 对分辨率的权衡

| 参数 | 变化 | 预期现象 |
|---|---|---|
| `n_fft` 从 512 → 2048 | 频率分辨率 ↑（`sr/n_fft`：31 Hz → 7.8 Hz） | 谐波条纹更细，但时间分辨率下降（水平模糊） |
| `hop_length` 从 256 → 64 | 时间分辨率 ↑（4× 更多帧） | 爆破辅音变细，矩阵变宽 4 倍 |
| `n_mels` 从 40 → 128 | Mel 频带密度 ↑ | log-Mel 图纵向更精细，高频共振峰更清晰 |

观察要点：`n_fft * hop_length` 的乘积固定时，两个分辨率此消彼长。语音 ASR 常用 `n_fft=512, hop=160`（25ms帧 / 10ms步长）；音乐分析常用更大的 `n_fft=2048`。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

configs = [
    dict(n_fft=512,  hop_length=128,  label="n_fft=512, hop=128\n（时间分辨率高）"),
    dict(n_fft=1024, hop_length=256,  label="n_fft=1024, hop=256\n（均衡，默认）"),
    dict(n_fft=2048, hop_length=512,  label="n_fft=2048, hop=512\n（频率分辨率高）"),
]

for ax, cfg in zip(axes, configs):
    label = cfg.pop("label")
    _mag = magnitude_spectrogram(samples, **cfg)
    _freqs = np.linspace(0, sr / 2, _mag.shape[1])
    _f_idx = np.searchsorted(_freqs, 6000)
    _t = np.arange(_mag.shape[0]) * cfg["hop_length"] / sr
    ax.imshow(
        20 * np.log10(_mag[:, :_f_idx].T + 1e-8),
        aspect="auto", origin="lower",
        extent=[_t[0], _t[-1], 0, _freqs[_f_idx - 1]],
        cmap="magma",
    )
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("时间 (s)")
    ax.set_ylabel("频率 (Hz)")
      # restore (already consumed by pop)

plt.suptitle("n_fft 对频谱图分辨率的影响", fontsize=12)
plt.savefig("L51_nfft_compare.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 参数对比图保存至 L51_nfft_compare.png")


## 练习（TODO）

本节学到了完整的 WAV → STFT → Mel → MFCC 流水线。下面 3 道练习帮你把概念转化为动手能力。

**练习 1**：改变帧移，验证帧数变化

目标：将 `hop_length` 改为 160（10 ms 帧移 @ 16 kHz），重新提取 MFCC，验证帧数 ≈ `duration / 0.01`。

**练习 2**：修改 n_mels，观察 MFCC 变化

目标：将 `n_mels` 从 80 改为 40，重新提取 MFCC，验证 coeffs 的形状第 1 维不变（帧数一致），并画出两组 MFCC 做对比。

**练习 3**：用不同信号运行流水线

目标：在 `make_vcv_signal()` 中把元音 /a/ 基频从 120 Hz 改为 220 Hz，重新运行流水线，在频谱图中找到谐波条纹位置是否上移。


In [ ]:
# ── 练习 1：10 ms 帧移，验证帧数 ──

hop_10ms = 160  # 10 ms @ 16 kHz

# TODO: 用 hop_10ms 调用 mfcc()，将结果存入 coeffs_10ms
# （提示：与 cell 8 中调用方式相同，只换 hop_length）
raise NotImplementedError("练习 1：请填写 mfcc() 调用，用 hop_length=hop_10ms")

# 验证帧数近似等于 duration / 0.01
expected_frames_approx = int(duration / 0.01)
actual_frames = coeffs_10ms.shape[0]
assert abs(actual_frames - expected_frames_approx) <= 5, (
    f"帧数偏差过大：期望 ≈{expected_frames_approx}，实际 {actual_frames}。"
    f"请检查 hop_length 是否设为 {hop_10ms}，signal 时长 {duration:.3f}s。"
)
print(f"✅ 练习 1 通过：hop=160，帧数={actual_frames}（期望 ≈{expected_frames_approx}）")

In [ ]:
# ── 练习 2：n_mels=40 vs n_mels=80，对比 MFCC ──

# TODO: 分别用 n_mels=40 和 n_mels=80 提取 MFCC（n_mfcc=13，其余参数与 cell 8 相同）
# 将结果存入 coeffs_mel40 和 coeffs_mel80
raise NotImplementedError("练习 2：请填写两次 mfcc() 调用，分别用 n_mels=40 和 n_mels=80")

# shape 验证：帧数应相同
assert coeffs_mel40.shape[0] == coeffs_mel80.shape[0], (
    f"帧数不一致：n_mels=40 得到 {coeffs_mel40.shape[0]} 帧，"
    f"n_mels=80 得到 {coeffs_mel80.shape[0]} 帧。hop_length 是否相同？"
)
assert coeffs_mel40.shape[1] == coeffs_mel80.shape[1] == 13, (
    "n_mfcc 维度应均为 13"
)
print(f"✅ 练习 2 通过：n_mels=40 形状={coeffs_mel40.shape}，n_mels=80 形状={coeffs_mel80.shape}")

In [ ]:
# ── 练习 3：修改 F0，观察谐波上移 ──

def make_vcv_f0(f0: float = 220, sr: int = 16000) -> np.ndarray:
    """合成与 make_vcv_signal 相同结构的 V-C-V，但元音 /a/ 基频可调。"""
    # TODO: 复制 make_vcv_signal 中的代码，把 120 替换为 f0 参数
    raise NotImplementedError("练习 3：请实现 make_vcv_f0，把基频改为参数 f0")

signal_f0_220 = make_vcv_f0(f0=220, sr=sr)

# 验证：提取 MFCC，形状应与 coeffs 一致（同等时长、相同 hop）
coeffs_f0_220 = mfcc(signal_f0_220, sr, n_mfcc=13, n_fft=1024, hop_length=256, n_mels=80)
assert coeffs_f0_220.shape == coeffs.shape, (
    f"形状不匹配：f0=220 得到 {coeffs_f0_220.shape}，期望 {coeffs.shape}。"
    "请确认 make_vcv_f0 生成的信号时长与 make_vcv_signal 相同。"
)
print(f"✅ 练习 3 通过：F0=220 Hz MFCC 形状={coeffs_f0_220.shape}，与原始一致")
print("  提示：在 cell 8 的幅度频谱图上可观察到谐波条纹从 ~120 Hz 间距变为 ~220 Hz 间距")

## 本课收束

本节用 `read_wav` 加载 WAV 文件（float64, [-1,1]），依次调用 `magnitude_spectrogram`、`mel_spectrogram`、`mfcc`，输出形状分别为 `(T, n_fft//2+1)`、`(T, n_mels)`、`(T, n_mfcc)`。
可视化验证了元音的谐波条纹与辅音的宽带瞬态，并通过可选 librosa 对比确认 Aurora Audio Core 的数值正确性。
所有计算来自 `aurora.audio`（`io.py / stft.py / mel.py / mfcc.py`），零依赖 librosa 或 scipy.signal，符合 CLAUDE.md 中 Audio Core 的原则。
下一课（L52）是 Audio Core 完结：38 个单元测试全绿，提取面试证据链，验证 MFCC 流水线端到端输出正确。


---

→ **下一课**　[L52 · Audio Core 完结](L52_features_done.ipynb)

> 下节课将学习 **Audio Core 完结**：特征工程收口，38 个单元测试全绿，面试证据整理。